# Instructor Notebook 07 — Agentic Systems: Reflection, Live
**ComplianceGPT Lab · REU 2026 · Week 3 (bonus) / Week 4 bridge**

> **Teaching script:** Part 1 runs entirely on real experiment CSVs already in the repo — no model inference required, safe to run cold in front of the room. Part 2 is optional live coding that needs Ollama running; skip it if the network/GPU isn't cooperating and just talk through the code.

**Learning arc:**
1. What `extract_with_reflection()` actually does, pass by pass (real code, not a diagram)
2. The real ablation numbers: single-pass vs. +RAG vs. +reflection, on Qwen2.5:72B
3. The surprise: RAG *hurts* Qwen, reflection undoes the damage — verified row by row
4. Contrast with Gemma3:4B (Week 2 data): same architecture, opposite lesson
5. (Optional, live) Build a one-step reflective wrapper and run it on a real failing case
6. What students do for the assignment

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

FINALRUN = '/Users/priscilladanso/Documents/GitHub/COMPLIANCEGPT/experiments/finalserverrun/'

singlepass = pd.read_csv(FINALRUN + 'ablation_qwen25_72b_singlepass_no_rag.csv')
rag_only   = pd.read_csv(FINALRUN + 'ablation_qwen25_72b_rag_no_reflect.csv')
reflect4   = pd.read_csv(FINALRUN + 'final_vast_qwen25_72b_v3.csv')   # the full 4-pass pipeline

print('Loaded three real Qwen2.5:72B runs, same 137 GoldCoin-HHS scenarios:')
print(f'  Single-pass (no RAG):   {len(singlepass)} rows')
print(f'  + BM25 RAG, no reflect: {len(rag_only)} rows')
print(f'  + Four-pass reflection: {len(reflect4)} rows')

---
## Part 1 · The Pipeline You're Researching, in Three Configurations

> **Say:** "Every one of these three files came from `app/batch_runner.py`, calling one of three methods in `connector/llm1_extractor.py`: `extract()`, `extract_with_rag()`, and `extract_with_reflection()`. Same 137 scenarios, same model, same GPU. The only thing that changed is how many times — and how — the LLM was asked to look at its own answer."

```
extract()                    — 1 LLM call.  Scenario in, JSON facts out.
extract_with_rag()            — 1 LLM call, but BM25-retrieved predicate vocabulary is injected first.
extract_with_reflection()     — extract_with_rag() then 3 MORE LLM calls that re-examine the first pass's own facts:
                                   Pass 1.5 — CI-tuple check   (sender/receiver/purpose direction)
                                   Pass 2   — Oracle check     (7 highest-impact boolean predicates)
                                   Pass 3   — Judicial check   (only fires for §164.512(e)/(f) scenarios)
```

In [ ]:
def summarize(name, df):
    n = len(df)
    acc = (df['match'] == 'Y').mean() * 100
    return {'Configuration': name, 'n': n, 'Accuracy': round(acc, 1), 'Correct': int((df['match']=='Y').sum())}

rows = [
    summarize('Single-pass (no RAG)', singlepass),
    summarize('+ BM25 RAG (no reflect)', rag_only),
    summarize('+ Four-pass reflection', reflect4),
]
summary = pd.DataFrame(rows)
summary['Delta vs. prev'] = summary['Accuracy'].diff().round(1)
print(summary.to_string(index=False))

> **Say:** "Before I show you the numbers — predict out loud. Does adding retrieval help? Does adding reflection help more? For Gemma3:4B last week the answer was yes and yes. Watch what happens on the *bigger* model."

Run the cell above, then let the room react. The expected pattern:

| Configuration | Accuracy |
|---|---|
| Single-pass (no RAG) | 92.7% |
| + BM25 RAG (no reflect) | **84.7%** ← *worse* |
| + Four-pass reflection | 92.7% ← *back to baseline, not above it* |

---
## Part 1b · Why? Find the Actual Rows That Flipped

> **Say:** "Don't take the aggregate number on faith. Let's find the specific scenarios where adding RAG turned a right answer into a wrong one."

In [ ]:
# Align by row_id and find where single-pass was right but +RAG was wrong
merged = singlepass[['row_id', 'question', 'ground_truth', 'verdict_norm', 'match']].merge(
    rag_only[['row_id', 'verdict_norm', 'match']],
    on='row_id', suffixes=('_single', '_rag')
)

regressions = merged[(merged['match_single'] == 'Y') & (merged['match_rag'] == 'N')]
print(f'{len(regressions)} scenarios that RAG broke (single-pass was right, +RAG was wrong):\n')
for _, row in regressions.head(3).iterrows():
    print(f"row_id {row['row_id']}: truth={row['ground_truth']}  single-pass={row['verdict_norm_single']}  +RAG={row['verdict_norm_rag']}")
    print(f"  {str(row['question'])[:160]}...\n")

In [ ]:
# Now check: does the four-pass reflection pipeline recover these same rows?
merged2 = merged.merge(reflect4[['row_id', 'verdict_norm', 'match']], on='row_id')
merged2 = merged2.rename(columns={'verdict_norm': 'verdict_norm_reflect', 'match': 'match_reflect'})

recovered = merged2[(merged2['match_rag'] == 'N') & (merged2['match_reflect'] == 'Y')]
still_wrong = merged2[(merged2['match_rag'] == 'N') & (merged2['match_reflect'] == 'N')]
print(f'Of the rows +RAG got wrong: {len(recovered)} were fixed by reflection, {len(still_wrong)} were still wrong.')
print('\nThis is the concrete evidence for the claim: reflection\'s job here is damage control on RAG,')
print('not a new source of capability the single-pass model didn\'t already have.')

---
## Part 1c · Contrast With Gemma3:4B (Week 2 Data) — Same Architecture, Opposite Lesson

> **Say:** "This is the same three-way ablation, but on the small model you all profiled in Week 2. Same code, same flags, completely different story."

In [ ]:
comparison = pd.DataFrame([
    {'Configuration': 'Single-pass (no RAG)',   'Qwen2.5:72B': 92.7, 'Gemma3:4B': 54.0},
    {'Configuration': '+ BM25 RAG (no reflect)', 'Qwen2.5:72B': 84.7, 'Gemma3:4B': 63.5},
    {'Configuration': '+ Four-pass reflection',  'Qwen2.5:72B': 92.7, 'Gemma3:4B': 94.2},
    {'Configuration': 'Oracle ceiling',          'Qwen2.5:72B': 100.0, 'Gemma3:4B': 100.0},
])
print(comparison.to_string(index=False))
print()
print('Gemma3:4B:   54.0 -> 63.5 (+9.5pp) -> 94.2 (+30.7pp)   the pipeline is the whole story')
print('Qwen2.5:72B: 92.7 -> 84.7 (-8.0pp) -> 92.7 (+8.0pp)    the pipeline mostly undoes its own damage')

> **Discussion prompt for the room:** "If you only had budget to run ONE of these three configurations on a brand-new 72B-class model you'd never tested before, which would you pick, and what would you look at in the first 10 rows to decide whether to add the other two?"

---
## Part 2 · (Optional, Live) Build a One-Step Reflective Wrapper

> **Teaching note:** Requires Ollama running locally with a small model pulled (e.g. `gemma3:4b` or `qwen2.5:7b`). If it's not available, talk through the code instead of running it — the point is the *shape* of the wrapper, not a specific number.

This mirrors `extract_with_reflection()`'s Pass 2 (oracle check) in miniature: one extraction call, then one more call that re-examines the first call's own output.

In [ ]:
import requests, json

OLLAMA_URL = 'http://localhost:11434/api/generate'
MODEL = 'gemma3:4b'   # swap for whatever is pulled locally

def call_ollama(prompt, temperature=0.0, model=MODEL):
    r = requests.post(OLLAMA_URL, json={
        'model': model, 'prompt': prompt, 'stream': False,
        'options': {'temperature': temperature},
    }, timeout=120)
    r.raise_for_status()
    return r.json()['response']

def extract_json(text):
    start, end = text.find('{'), text.rfind('}')
    return json.loads(text[start:end+1]) if start != -1 and end != -1 else None

In [ ]:
# The same row_id 40 case from week3_prompting.ipynb
CASE_TEXT = (
    "In 2012, an individual found himself requiring emergency medical services and was "
    "taken by ambulance to a hospital. To receive this necessary medical attention, he provided the "
    "ambulance service, operated by the Philadelphia Fire Department Emergency Medical Services (EMS), "
    "with his personal and sensitive"
)
GROUND_TRUTH_VERDICT = 'DENIED'
SCHEMA = '{"sender_role": str, "receiver_role": str, "purpose": str, "is_business_associate": bool, "is_required_by_law": bool, "has_ba_agreement": bool}'

extract_prompt = f"""You are extracting facts from a HIPAA scenario. Output ONLY a JSON object matching this schema:
{SCHEMA}

Scenario: {CASE_TEXT}

JSON:"""

pass1_response = call_ollama(extract_prompt)
pass1_facts = extract_json(pass1_response)
print('Pass 1 (single-shot extraction):')
print(json.dumps(pass1_facts, indent=2))

In [ ]:
# One reflection pass: hand the model its own facts and ask it to double-check the riskiest field
reflect_prompt = f"""Scenario: {CASE_TEXT}

You previously extracted these facts:
{json.dumps(pass1_facts, indent=2)}

Re-examine ONLY the field \"is_required_by_law\". This should be true ONLY if the scenario
explicitly mentions a law, statute, subpoena, or court order requiring disclosure.
Does the scenario text above contain such explicit language? Answer with the corrected
full JSON object matching the same schema:
{SCHEMA}

Corrected JSON:"""

pass2_response = call_ollama(reflect_prompt)
pass2_facts = extract_json(pass2_response)
print('Pass 2 (one reflection step):')
print(json.dumps(pass2_facts, indent=2))
print()
print('Did reflection change is_required_by_law?',
      pass1_facts.get('is_required_by_law'), '->', pass2_facts.get('is_required_by_law') if pass2_facts else '(parse failed)')

---
## Part 3 · What Students Do Next

> **Say:** "You just watched a 2-pass reflective wrapper live. The assignment asks you to scale this to 15 real scenarios and measure it properly, the same way we just measured Qwen's 137-row ablation above."

**Assignment — Reflective Wrapper Mini-Lab** (see `slides-agentic-systems.html`, slide 15, and `assignments.html`):

1. Pick 15 GoldCoin-HHS scenarios, including the failing case you used in Week 3's prompting exercise.
2. Run each through baseline `extract()` — one pass, no reflection.
3. Run each through your own one-step reflection wrapper (built above).
4. Report: accuracy before vs. after, and for every case that flipped (right→wrong or wrong→right), one sentence on why — grounded in the actual scenario text, not a guess.

---
## Recap

1. `extract_with_reflection()` is 4 LLM calls, not 1 — RAG extraction, then CI-tuple / oracle / judicial re-checks
2. On Qwen2.5:72B, raw RAG *cost* 8.0pp of accuracy; reflection recovered exactly that much, landing back at the no-RAG baseline
3. On Gemma3:4B, the same pipeline was the dominant source of a +30.7pp gain — reflection's value is model-dependent
4. This is reflection (self re-prompting), not tool-calling (ReAct) — no live call to the Souffle verifier happens mid-loop
5. Next: the assignment scales this from 1 live-coded example to 15 scenarios, measured properly